# Phase 1: Data Exploration and Preprocessing

**Project:** A comparative study of RAG techniques for generating image captions

**Dataset:** MS COCO (train2017 and val2017)

---

## 1. Project Idea

### 1.1  What is the problem?
Image captioning means making the computer write a short sentence that describes an image's content.
Modern AI models can write good captions, but they often make mistakes:
- They **miss objects** that are in the image.
- They **invent objects** that are not in the image (hallucination).
- They are not precise about **colors, details and relations** between objects.

### 1.2 How can RAG solve this problem?
**RAG (Retrieval-Augmented Generation)** helps the model by giving it extra information before it writes the caption. It works in three steps:

1. **Retrieve:** search a database for images and captions similar to our image.
2. **Augment:** add the retrieved information to the model's prompt.
3. **Generate:** the model writes the caption using the image and the extra context.


### 1.3 What is the goal of this project?
We want to compare different RAG techniques on the same data. We measure:

- Caption quality
- Hallucination rate
- Execution time

---

## 2. The Dataset: MS COCO

### 2.1 What is COCO?
**MS COCO (Common Objects in Context)** is a large public dataset. It contains everyday photos where objects appear in natural scenes. It is one of the most used datasets for object detection, segmentation and image captioning.

COCO gives us three kinds of information for each image:
- **Objects:** each object has a category, a bounding box and a segmentation mask.
- **Captions:** about 5 sentences written by different humans that describe the image.
- **Categories:** 80 object classes grouped into 12 super-categories.

### 2.2 Structure
COCO has two parts: the **images** (`train2017/`, `val2017/`) and the **annotation files** (JSON). The `image_id` links an image to its objects and its captions.

| File | Content |
|------|---------|
| `instances_*.json` | objects: category, box, mask |
| `captions_*.json` | human captions |

### 2.3 Splits used in this project

| COCO split | Size (full) | How we use it |
|------------|-------------|---------------|
| **train2017** | about 118,000 images | We take **10,000 images** for the knowledge base and the validation set |
| **val2017** | 5,000 images | We take a **100-image** subset as the final test set |

Each image has about 5 human captions, so 10,000 images give about 50,000 captions.

## 3. Phase 1: Data Exploration and Preprocessing

### 3.1 What is the objective
Before building any RAG system, we need clean data that is the same for every group.
Phase 1 turns the raw COCO data into a clean dataset, fixed splits, and embeddings ready for retrieval.

### 3.2 Data Splits

| Set | Size | Source | Purpose |
|-----|------|--------|---------|
| **Knowledge base** | 10,000 images | train2017 | The database that retrieval searches in |
| **Validation** | 100 images | train2017 | Tune settings during development |
| **Test** | 100 images | val2017 | Final evaluation only |

**Anti-leakage rule:** test images and their captions must never appear in the knowledge base.

### 3.3 Work plan
We are 4 members. The work is split into 4 steps, and each step has 4 tasks.

| Step | Focus | Output |
|------|-------|--------|
| **Step 1** | Ingestion and environment setup | Raw data, repo, integrity audit, count check |
| **Step 2** | Splits, anti-leakage and metadata | Split JSON files, leakage checks, object tags |
| **Step 3** | Transformation and statistics | Clean images, clean captions, plots |
| **Step 4** | Vectorization, indexing and QA | Embeddings (.npy), FAISS index, final checks |

---

## Step 1: Ingestion and Environment Setup

### Task 1.1: Download the COCO data
**Goal:** get the 10,000 training images and their captions, the same for all team members.

**Script:** `scripts/download_coco_rag_subset.py`

**Before running:** download and extract `annotations_trainval2017.zip` from the COCO website.
The script needs the file `annotations/captions_train2017.json`.

**What the script does:**
1. Reads `captions_train2017.json` (about 118,000 images).
2. **Sorts** the images by id, then picks **10,000 random images** with a fixed seed (`20261027`).
   Because of the sort and the fixed seed, every team member gets the **same** images.
3. Collects the 5 captions of each selected image.
4. Saves the selection in `train_subset_10000.json`, so a rerun never changes the dataset.
5. Downloads the images into `train2017_subset/`.
   It retries up to 4 times, and it skips files that already exist, so we can run it again after a failure.

**Output files:**

| File | Content |
|------|---------|
| `train_subset_10000.json` | The frozen selection: id, file name, url, captions |
| `train_subset_10000_filenames.txt` | One file name per line |
| `captions_metadata_10000.json` | Metadata of the images requested |
| `train2017_subset/` | The downloaded images |
| `failed_downloads.json` | Only created if some downloads failed |

In [ ]:
!python scripts/download_coco_rag_subset.py --annotations-dir . --output-dir coco_rag_data

### Task 1.2: Repository setup
**Goal:** give the team one shared place for code and a common environment.

What we did:
- Created a shared **GitHub repository** for the project, so all 4 members work on the same code.
- Added a **`.gitignore`** file, so large files (images, embeddings) are not pushed to github.
- Added a **`requirements.txt`** file with the Python libraries, so everyone installs the same versions.

### Task 1.3: File integrity audit
**Goal:** make sure every selected image exists and can be opened before we use it.

**Script:** `scripts/diagnose_images.py`

What the script does:
1. Reads the expected file names from `train_subset_10000.json`.
2. Compares them with the files in `train2017_subset/`:
   - **missing** images: expected but not on disk,
   - **unexpected** images: on disk but not in the selection.
3. Tests every image in two ways:
   - `image.verify()` checks that the file structure is valid,
   - `image.load()` decodes all the pixels, which catches broken files that `verify()` misses.
4. Saves a report in `diagnostic_report.json` with the final status: `PASS` or `ISSUES_FOUND`.

The script only reports problems. It does not delete or fix anything.

In [ ]:
!python scripts/diagnose_images.py \
    --images-dir coco_rag_data/train2017_subset \
    --selection-file coco_rag_data/train_subset_10000.json \
    --report coco_rag_data/diagnostic_report.json

### Task 1.4: Verify dataset counts
**Goal:** check that the knowledge base is complete and consistent.

What we check:
1. The selection contains exactly **10,000** images, with no duplicate ids.
2. Every selected image exists on disk.
3. Every image has at least 5 captions.

The validation and test sets are created in Step 2, so we check their counts there.